In [2]:
import numpy as np
import openmdao.api as om
import os

os.environ['OPENMDAO_REPORTS'] = 'none'

In [3]:
class satelliteDis1(om.ExplicitComponent):
    def setup(self):
        # Global Design Variable
        self.add_input('x', val=np.ones(5))
        
        # Coupling parameter
        self.add_input('u21', val=9.)

        # Coupling output
        self.add_output('u12', val=9.)

        self.counter = 0

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        x1 = inputs['x'][0]
        x2 = inputs['x'][1]
        x3 = inputs['x'][2]
        u21 = inputs['u21']
        self.counter += 1
        # print('dis1')

        if u21.real < 0.0:
            u21 *= -1

        outputs['u12'] = x1**2 + 2*x2 - x3 + 2*(u21**0.5)
        # print('dis1', outputs['u12'])
        with open("output.txt", "a") as text_file:
            text_file.write("dis1 %s\n" % outputs['u12'])

class satelliteDis2(om.ExplicitComponent):
    def setup(self):
        # Global Design Variable
        self.add_input('x', val=np.ones(5))
        
        # Coupling parameter
        self.add_input('u12', val=9.)

        # Coupling output
        self.add_output('u21', val=9.)

        self.counter = 0
        
    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        x1 = inputs['x'][0]
        x4 = inputs['x'][3]
        x5 = inputs['x'][4]
        u12 = inputs['u12']
        self.counter += 1
        # print('dis2')

        outputs['u21'] = x1*x4 + x4**2 + x5 + u12
        with open("output.txt", "a") as text_file:
            text_file.write("dis2 %s\n" % outputs['u21'])

In [4]:
class satelliteGroup(om.Group):
    def setup(self):
        cycle = self.add_subsystem('cycle', om.Group(), promotes=['*'])
        cycle.add_subsystem('d1', satelliteDis1(), promotes_inputs=['x', 'u21'], 
                           promotes_outputs=['u12'])
        cycle.add_subsystem('d2', satelliteDis2(), promotes_inputs=['x', 'u12'], 
                           promotes_outputs=['u21'])

        nlbgs = cycle.nonlinear_solver = om.NonlinearBlockGS()
        # nlbgs = cycle.nonlinear_solver = om.NewtonSolver(solve_subsystems=True, iprint=2)
        nlbgs.options['maxiter'] = 1000
        nlbgs.options['iprint'] = 2

        self.add_subsystem('obj_cmp', om.ExecComp('f = ((x[0]**0.5 + x[3] + 0.4*x[0]*x[4]) - (4.5 - (x[0]**2 + 2*x[1] + x[2] + x[1]*exp(-u21))))', x=np.ones(5), u21=9.0),
                           promotes_inputs=['x','u21'], promotes_outputs=['f'])
        self.add_subsystem('res1', om.ExecComp('r1 = u12 - (x[0]**2 + 2*x[1] - x[2] + 2*(u21**0.5))', x=np.ones(5)*0.5, u21=9.0),
                           promotes_inputs=['x','u21'], promotes_outputs=['r1'])
        self.add_subsystem('res2', om.ExecComp('r2 = u21 - (x[0]*x[3] + x[3]**2 + x[4] + u12)', x=np.ones(5), u12=9.0),
                           promotes_inputs=['x','u12'], promotes_outputs=['r2'])    

        open('output.txt', 'w').close()

In [5]:
prob = om.Problem()
prob.model = satelliteGroup()

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'COBYQA'
prob.driver.options['tol'] = 1e-8
prob.driver.options['disp'] = False

prob.model.add_design_var('x', lower=np.ones(5)*0, upper=np.ones(5)*2)
prob.model.add_objective('f')
prob.model.add_constraint('r1', lower=0., upper=0.)
prob.model.add_constraint('r2', lower=0., upper=0.)
# prob.model.add_constraint('u21', lower=0.)
prob.model.add_constraint('u12', lower=0.)

prob.model.set_input_defaults('x', np.zeros(5))

prob.model.approx_totals()

prob.setup()

prob.set_val('x',np.ones(5))

prob.run_model()

# prob.run_driver()

print('min objective', prob.get_val('f'))

print('residuals')
print('r1', prob.get_val('r1'))
print('r2', prob.get_val('r2'))

print('design vars')
print('x', prob.get_val('x'))

print('coupling vars')
print('u12', prob.get_val('u12'))
print('u21', prob.get_val('u21'))


=====
cycle
=====
NL: NLBGS 1 ; 2.23606798 1
NL: NLBGS 2 ; 0.895550145 0.4005022
NL: NLBGS 3 ; 0.266240401 0.119066327
NL: NLBGS 4 ; 0.0777458045 0.0347689808
NL: NLBGS 5 ; 0.0225858508 0.0101006995
NL: NLBGS 6 ; 0.0065515879 0.00292995918
NL: NLBGS 7 ; 0.00189962782 0.000849539389
NL: NLBGS 8 ; 0.000550726444 0.000246292353
NL: NLBGS 9 ; 0.000159656839 7.14007091e-05
NL: NLBGS 10 ; 4.62843896e-05 2.06990083e-05
NL: NLBGS 11 ; 1.34177663e-05 6.00060753e-06
NL: NLBGS 12 ; 3.88978434e-06 1.73956444e-06
NL: NLBGS 13 ; 1.12764062e-06 5.04296216e-07
NL: NLBGS 14 ; 3.26900713e-07 1.46194443e-07
NL: NLBGS 15 ; 9.47678466e-08 4.23814694e-08
NL: NLBGS 16 ; 2.7473003e-08 1.22863005e-08
NL: NLBGS 17 ; 7.96436811e-09 3.5617737e-09
NL: NLBGS 18 ; 2.30885487e-09 1.03255129e-09
NL: NLBGS 19 ; 6.69331671e-10 2.99334223e-10
NL: NLBGS 20 ; 1.94038306e-10 8.67765687e-11
NL: NLBGS Converged
min objective [1.9000068]
residuals
r1 [-7.89897949]
r2 [-10.89897949]
design vars
x [1. 1. 1. 1. 1.]
coupling vars

In [6]:
# Run problem, save residuals
import sys
import shutil
import torch
from scipy.stats import qmc
from botorch.utils.transforms import unnormalize

prob = om.Problem()
prob.model = satelliteGroup()

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'COBYQA'
prob.driver.options['tol'] = 1e-8
prob.driver.options['disp'] = False

prob.model.add_design_var('x', lower=np.ones(5)*0, upper=np.ones(5)*2)
prob.model.add_objective('f')
prob.model.add_constraint('r1', lower=0., upper=0.)
prob.model.add_constraint('r2', lower=0., upper=0.)
# prob.model.add_constraint('u21', lower=0.)
prob.model.add_constraint('u12', lower=0.)

prob.model.set_input_defaults('x', np.ones(5))

prob.model.approx_totals()

# prob.setup()

# prob.set_val('x',np.ones(5))
dtype = torch.float64
device= "cpu"
bounds  = torch.tensor([[0., 0., 0., 0., 0.], 
                        [2., 2., 2., 2., 2.]], dtype=dtype, device=device) # always a 2 x d tensor
# Generate input points
npts = 10
sampler = qmc.LatinHypercube(d=5, seed=1234)
x_input = unnormalize(torch.tensor(sampler.random(n=npts)), bounds=bounds)
for i, x in zip(range(0,npts), x_input):
    # prob.set_val('B', 233.1)
    prob.setup() # call here to clear output.txt
    prob.set_val('x', x)

    prob.run_model()

    shutil.copy('output.txt', 'satellite_output' + str(i+1) + '.txt')

    # stdout = sys.stdout
    # with open('satellite_res_' + str(i+1) + '.txt', 'w') as sys.stdout:
    #     
    # sys.stdout = stdout

# prob.run_driver()

# print('min objective', prob.get_val('f'))

# print('residuals')
# print('r1', prob.get_val('r1'))
# print('r2', prob.get_val('r2'))

# print('design vars')
# print('x', prob.get_val('x'))

# print('coupling vars')
# print('u12', prob.get_val('u12'))
# print('u21', prob.get_val('u21'))


=====
cycle
=====
NL: NLBGS 1 ; 1.01547829 1
NL: NLBGS 2 ; 0.146810577 0.144572837
NL: NLBGS 3 ; 0.0499480578 0.0491867314
NL: NLBGS 4 ; 0.0170621478 0.0168020803
NL: NLBGS 5 ; 0.00583646301 0.00574750152
NL: NLBGS 6 ; 0.00199742995 0.00196698439
NL: NLBGS 7 ; 0.000683697239 0.000673276077
NL: NLBGS 8 ; 0.000234034678 0.000230467436
NL: NLBGS 9 ; 8.01133433e-05 7.88922266e-05
NL: NLBGS 10 ; 2.74240963e-05 2.70060882e-05
NL: NLBGS 11 ; 9.3877337e-06 9.24464245e-06
NL: NLBGS 12 ; 3.21358305e-06 3.16460045e-06
NL: NLBGS 13 ; 1.10006516e-06 1.08329757e-06
NL: NLBGS 14 ; 3.76571397e-07 3.70831559e-07
NL: NLBGS 15 ; 1.2890693e-07 1.26942083e-07
NL: NLBGS 16 ; 4.41270793e-08 4.34544783e-08
NL: NLBGS 17 ; 1.51054676e-08 1.48752246e-08
NL: NLBGS 18 ; 5.17086211e-09 5.09204596e-09
NL: NLBGS 19 ; 1.77007702e-09 1.74309686e-09
NL: NLBGS 20 ; 6.0592757e-10 5.96691801e-10
NL: NLBGS 21 ; 2.07420518e-10 2.0425894e-10
NL: NLBGS 22 ; 7.10033492e-11 6.99210904e-11
NL: NLBGS Converged

=====
cycle
=====


In [7]:
prob.model.cycle.d1.counter

18

In [8]:
prob.model.cycle.d2.counter

18

In [8]:
x_input

tensor([[0.0047, 1.1240, 0.2154, 0.3477, 0.5362],
        [1.5764, 0.5516, 0.9363, 0.0072, 1.9473],
        [1.9118, 0.2780, 0.4273, 1.6272, 1.0650],
        [0.2680, 0.6528, 1.9554, 1.5656, 0.0259],
        [0.7880, 1.6633, 1.6658, 1.2778, 0.3880],
        [1.0044, 0.1122, 0.6935, 1.1994, 1.3497],
        [0.8283, 1.5149, 1.2528, 1.8156, 0.7693],
        [1.6015, 0.9635, 1.0120, 0.9826, 1.7064],
        [0.4342, 1.3438, 1.4262, 0.4047, 1.4317],
        [1.3102, 1.9259, 0.1035, 0.7816, 0.9547]], dtype=torch.float64)

In [13]:
prob.list_problem_vars()

----------------
Design Variables
----------------
name  val           size  indices  
----  ------------  ----  ------- 
x     |2.63793552|  5     None     

-----------
Constraints
-----------
name  val             size  indices  alias  
----  --------------  ----  -------  ----- 
r1    [-12.48317683]  1     None     None   
r2    [-15.07270651]  1     None     None   
u12   [13.48317682]   1     None     None   

----------
Objectives
----------
name  val           size  indices  
----  ------------  ----  ------- 
f     [3.59849728]  1     None     



C:\Users\kaily\miniconda3\envs\bayesian_opt\Lib\site-packages\openmdao\core\problem.py:1889: OMDeprecationWarning:Method `list_problem_vars` has been renamed `list_driver_vars`.
Please update your code to use list_driver_vars to avoid this warning.


{'design_vars': [('x',
   {'name': 'x',
    'indices': None,
    'size': np.int32(5),
    'val': array([1.31022528, 1.92589831, 0.10346611, 0.78156369, 0.95466338])})],
 'constraints': [('r1',
   {'name': 'r1',
    'alias': None,
    'indices': None,
    'size': np.int32(1),
    'val': array([-12.48317683])}),
  ('r2',
   {'name': 'r2',
    'alias': None,
    'indices': None,
    'size': np.int32(1),
    'val': array([-15.07270651])}),
  ('u12',
   {'name': 'u12',
    'alias': None,
    'indices': None,
    'size': np.int32(1),
    'val': array([13.48317682])})],
 'objectives': [('f',
   {'name': 'f',
    'indices': None,
    'size': np.int32(1),
    'val': array([3.59849728])})]}

In [15]:
print('x', prob.get_val('x'))
print('u12', prob.get_val('u12'))
print('u21', prob.get_val('u21'))

x [1.31022528 1.92589831 0.10346611 0.78156369 0.95466338]
u12 [13.48317682]
u21 [16.07270651]
